## STAT3612 Group17 Track 1

In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
DATA_DIR = Path("/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project")

# 结构化表格（每行一个住院样本，含标签/静态信息）
train_df = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/train.csv")   # 至少包含 id, label, 性别, 年龄, ...
valid_df  = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/valid.csv")
test_df = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/test.csv")

#print(train_df.info())
#print(test_df.info())

# 时序特征字典：key 为 id，value 为 (n_days, 171) 的 numpy 数组
ehr_data = np.load(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/ehr_preprocessed_seq_by_day_cat_embedding.pkl", allow_pickle=True)

feat_dict = ehr_data["feat_dict"]
feature_cols: List[str] = ehr_data["feature_cols"]

print(type(ehr_data))
print(type(feat_dict), len(feat_dict))
print(list(feat_dict.keys())[:5])        # 只看前几个病人 ID
print(feature_cols[:10])                 # 只看前 10 个特征名
print(feat_dict[next(iter(feat_dict))].shape)  # 查看单个样本的数组尺寸

<class 'dict'>
<class 'dict'> 14532
['10869829_25238191', '17910612_22301530', '16026764_21404901', '12347278_29852086', '18463717_24608289']
['age', 'gender', 'ethnicity', 'Y90-Y99', 'G30-G32', 'O85-O92', 'C60-C63', 'F40-F48', 'M80-M85', 'R00-R09']
(13, 171)


In [4]:
demo_cols = ehr_data["demo_cols"]  # 如果有不同列名，按实际修改

icd_cols = ehr_data["icd_cols"]
lab_cols = ehr_data["lab_cols"]
med_cols = ehr_data["med_cols"]

demo_idxs = [feature_cols.index(c) for c in demo_cols]
icd_idxs  = [feature_cols.index(c) for c in icd_cols]
lab_idxs  = [feature_cols.index(c) for c in lab_cols]
med_idxs  = [feature_cols.index(c) for c in med_cols]

print(len(demo_idxs), len(icd_idxs), len(lab_idxs), len(med_idxs))
print(icd_cols, lab_cols, med_cols)

3 91 36 41
['Y90-Y99', 'G30-G32', 'O85-O92', 'C60-C63', 'F40-F48', 'M80-M85', 'R00-R09', 'J90-J94', 'A00-A09', 'E00-E07', 'F01-F09', 'F30-F39', 'H30-H36', 'D60-D64', 'N00-N08', 'F60-F69', 'I80-I89', 'I95-I99', 'N30-N39', 'K55-K64', 'F50-F59', 'R40-R46', 'J60-J70', 'N20-N23', 'I30-I52', 'R50-R69', 'B25-B34', 'C00-C14', 'D65-D69', 'C73-C75', 'G35-G37', 'E70-E88', 'K20-K31', 'C30-C39', 'I70-I79', 'M60-M63', 'A20-A28', 'N10-N16', 'E89-E89', 'R70-R79', 'C7B-C7B', 'M50-M54', 'A30-A49', 'F20-F29', 'G89-G99', 'R30-R39', 'J30-J39', 'N25-N29', 'Q65-Q79', 'G20-G26', 'B20-B20', 'K65-K68', 'R10-R19', 'E65-E68', 'B65-B83', 'E40-E46', 'F70-F79', 'N17-N19', 'J09-J18', 'J40-J47', 'C15-C26', 'L80-L99', 'B50-B64', 'O60-O77', 'C7A-C7A', 'B85-B89', 'E50-E64', 'K00-K14', 'R20-R23', 'J00-J06', 'N60-N65', 'D37-D48', 'K35-K38', 'G00-G09', 'M05-M14', 'I10-I16', 'B35-B49', 'K40-K46', 'K70-K77', 'Q00-Q07', 'E20-E35', 'J20-J22', 'A80-A89', 'B00-B09', 'J80-J84', 'G60-G65', 'D3A-D3A', 'G10-G14', 'B90-B94', 'N40-N53'

### Data Processing

In [5]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Feature engineering utilities
# ---------------------------------------------------------------------

def _compute_slope(series: np.ndarray) -> float:
    """最小二乘拟合趋势线斜率（按时间顺序的变化率）。"""
    mask = ~np.isnan(series)
    if mask.sum() < 2:
        return 0.0

    y = series[mask].astype(float, copy=False)
    x = np.flatnonzero(mask).astype(float)

    if np.allclose(y, y.mean()):
        return 0.0

    return float(np.polyfit(x, y, 1)[0])


def summarize_static(series: np.ndarray, prefix: str, seq_len: int | None = None) -> dict[str, float]:
    """静态/缓慢变化特征：仅保留最后一个非缺失值。"""
    mask = ~np.isnan(series)
    if not mask.any():
        return {f"{prefix}__last": np.nan}

    last_value = float(series[mask][-1])
    return {f"{prefix}__last": last_value}


def summarize_lab(series: np.ndarray, prefix: str, seq_len: int | None = None) -> dict[str, float]:
    """
    动态实验室检查：计算均值、标准差、最值、首末值、变化量、斜率、观测次数。
    """
    mask = ~np.isnan(series)
    if not mask.any():
        return {
            f"{prefix}__mean": np.nan,
            f"{prefix}__std": np.nan,
            f"{prefix}__min": np.nan,
            f"{prefix}__max": np.nan,
            f"{prefix}__first": np.nan,
            f"{prefix}__last": np.nan,
            f"{prefix}__delta": np.nan,
            f"{prefix}__slope": 0.0,
            f"{prefix}__slope*days": 0.0,
        }

    values = series[mask].astype(float, copy=False)
    first_val = float(values[0])
    last_val = float(values[-1])
    slope = _compute_slope(series.astype(float, copy=False))
    effective_len = seq_len if seq_len is not None else mask.sum()

    return {
        f"{prefix}__mean": float(values.mean()),
        f"{prefix}__std": float(values.std(ddof=0)),
        f"{prefix}__min": float(values.min()),
        f"{prefix}__max": float(values.max()),
        f"{prefix}__first": first_val,
        f"{prefix}__last": last_val,
        f"{prefix}__delta": last_val - first_val,
        f"{prefix}__slope": slope,
        f"{prefix}__slope*days": slope * float(effective_len),
    }


def extract_patient_id(pid: str) -> str:
    """
    默认用第一个分隔符（优先 '_'，其次 '-', '__', '::'）前的部分作为 patient_id。
    若无分隔符，则直接返回原始 pid。
    """
    if not isinstance(pid, str):
        return str(pid)

    for sep in ("_", "-", "__", "::"):
        if sep in pid:
            return pid.split(sep, 1)[0]
    return pid


# ---------------------------------------------------------------------
# Group configuration
# ---------------------------------------------------------------------
# 预先准备好的列名与索引列表（需与 feat_dict 对应）
# demo_cols, demo_idxs, lab_cols, lab_idxs, icd_cols, icd_idxs, med_cols, med_idxs
# 在此假定这些列表以及 feat_dict 已经定义好。

group_meta = {
    "demo": {"cols": demo_cols, "idxs": demo_idxs, "agg": summarize_static},
    "icd":  {"cols": icd_cols,  "idxs": icd_idxs,  "agg": summarize_static},
    "med":  {"cols": med_cols,  "idxs": med_idxs,  "agg": summarize_static},
    "lab":  {"cols": lab_cols,  "idxs": lab_idxs,  "agg": summarize_lab},
}

# ---------------------------------------------------------------------
# Patient-level feature collapse
# ---------------------------------------------------------------------

patient_rows: list[dict[str, float]] = []

for pid, arr in feat_dict.items():
    arr = np.asarray(arr, dtype=np.float32)
    seq_len = arr.shape[0]

    row: dict[str, float] = {
        "pid": pid,
        "seq_len": float(seq_len),
    }

    for group_name, meta in group_meta.items():
        cols = meta["cols"]
        idxs = meta["idxs"]
        agg_fn = meta["agg"]

        if not cols:
            continue

        if len(cols) != len(idxs):
            raise ValueError(f"{group_name}: cols 与 idxs 长度不一致")

        block = arr[:, idxs]  # shape: (seq_len, num_features)

        for col_name, series in zip(cols, block.T):
            row.update(agg_fn(series, prefix=col_name, seq_len=seq_len))

    patient_rows.append(row)

feature_df = (
    pd.DataFrame(patient_rows)
      .set_index("pid")
      .astype(np.float32)
      .sort_index()
)

# 检查结果
print(feature_df.shape)
print(feature_df.head())

(14532, 460)
                   seq_len  age__last  gender__last  ethnicity__last  \
pid                                                                    
10001884_26184834     14.0       77.0           0.0              2.0   
10002013_23581541      6.0       57.0           0.0              6.0   
10002428_23473524     12.0       81.0           0.0              6.0   
10002428_28662225     18.0       81.0           0.0              6.0   
10003019_20962108      9.0       74.0           1.0              6.0   

                   Y90-Y99__last  G30-G32__last  O85-O92__last  C60-C63__last  \
pid                                                                             
10001884_26184834            0.0            0.0            0.0            0.0   
10002013_23581541            0.0            0.0            0.0            0.0   
10002428_23473524            0.0            0.0            0.0            0.0   
10002428_28662225            0.0            0.0            0.0            0.0

In [6]:
# === 1. 准备 train / valid 的标签 ======================================
train_labels = (
    train_df.loc[:, ["id", "readmitted_within_30days"]]
            .rename(columns={"readmitted_within_30days": "label"})
            .assign(split="train")
)

valid_labels = (
    valid_df.loc[:, ["id", "readmitted_within_30days"]]   # 如果变量叫 test_df，就改成 test_df
            .rename(columns={"readmitted_within_30days": "label"})
            .assign(split="valid")
)


label_df = pd.concat([train_labels, valid_labels], ignore_index=True)
label_df["id"] = label_df["id"].astype(str)
label_series = label_df.set_index("id")["label"].astype(np.int8)

print(label_df["split"].value_counts())  # 检查标签是否覆盖 train 和 valid

# === 2. 与特征表对齐 ================================================
feature_df.index = feature_df.index.astype(str)

dataset = feature_df.join(label_series, how="inner")

# 1. 如果 feature_df 的索引就是 id（pid），可以先把重复索引去掉：
feature_unique = feature_df.loc[~feature_df.index.duplicated(keep="first")]

# 2. 再与标签做 inner join
dataset = feature_unique.join(label_series, how="inner")

# --- 或者已经 join 完成，只想在 dataset 里去重 ----------------------
# 按索引（id/pid）保留第一条，其余全部丢弃
dataset = dataset.loc[~dataset.index.duplicated(keep="first")]

print(dataset.shape)
print(dataset.index.nunique())

print("dataset shape:", dataset.shape)
print("dataset index unique:", dataset.index.nunique())
print("缺少的 valid ids 数量：",
      np.setdiff1d(valid_df["id"].astype(str).unique(), dataset.index).size)

# === 3. 按官方 ID 划分 train / valid ================================
train_ids = np.intersect1d(train_df["id"].astype(str).unique(), dataset.index)
valid_ids = np.intersect1d(valid_df["id"].astype(str).unique(), dataset.index)

X_train = dataset.loc[train_ids].drop(columns="label")
y_train = dataset.loc[train_ids, "label"]

X_val   = dataset.loc[valid_ids].drop(columns="label")
y_val   = dataset.loc[valid_ids, "label"]

print(f"Train shape: {X_train.shape}, positives = {y_train.sum()}")
print(f"Valid shape: {X_val.shape}, positives = {y_val.sum()}")


split
train    49451
valid    16721
Name: count, dtype: int64
(11022, 461)
11022
dataset shape: (11022, 461)
dataset index unique: 11022
缺少的 valid ids 数量： 0
Train shape: (8234, 460), positives = 1449
Valid shape: (2788, 460), positives = 481


In [7]:
# === 获取 Kaggle 测试集特征矩阵 X_test =====================================

# 1. 保证特征表索引唯一且为字符串
feature_unique = feature_df.loc[~feature_df.index.duplicated(keep="first")].copy()
feature_unique.index = feature_unique.index.astype(str)

print(feature_unique.shape)

# 2. 取出测试集中出现的 ID（保持官方顺序）
test_ids = test_df["id"].astype(str).tolist()


# 4. 构建 X_test，并对齐训练集的列顺序
X_test = (
    feature_unique.reindex(test_ids)          # 按官方 ID 顺序排列
                 .reindex(columns=X_train.columns)  # 与训练特征列完全一致
)

print(f"X_test shape: {X_test.shape}")

# 1. 去掉 test_df 里重复的 id，保留第一次出现的行
test_df_unique = (
    test_df.copy()
           .assign(id=test_df["id"].astype(str))
           .loc[~test_df["id"].duplicated(keep="first")]
)

# 2. 取唯一的测试集 ID（顺序与官方 test.csv 一致）
test_ids = test_df_unique["id"].tolist()
print(f"Unique test ids: {len(test_ids)}")

# 3. 如果想再确认 feature_unique 也没有重复索引，可再次检查
assert feature_unique.index.is_unique, "feature_unique 仍然有重复索引，请先清理 feature_df"

# 4. 构建 X_test
X_test = (
    feature_unique.reindex(test_ids)          # 按唯一 ID 顺序排列
                  .reindex(columns=X_train.columns)
)

print(f"X_test shape after dedup: {X_test.shape}")
print(f"Missing test ids count: {(X_test.index.isna()).sum()}")

(14532, 460)
X_test shape: (16293, 460)
Unique test ids: 2741
X_test shape after dedup: (2741, 460)
Missing test ids count: 0


### 开始训练模型

### XGBoost

In [ ]:
import gc
import numpy as np
import optuna
import xgboost as xgb
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from sklearn.model_selection import StratifiedGroupKFold

# 假设 X_train, y_train, groups 已经定义
# 建议显式转换数据类型以节省内存，例如 float32
X_train = X_train.astype(np.float32)
print(f"X_train shape: {X_train.shape}")

groups = X_train.index
if hasattr(groups, "get_level_values"):
    groups = groups.get_level_values(0) 

neg_pos_ratio = (len(y_train) - y_train.sum()) / y_train.sum()

sampler = TPESampler(multivariate=True, n_startup_trials=15, seed=42)
# 使用 MedianPruner，并在前 1/3 的 trial 只有做完第1个 Fold 后才开始剪枝
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial: optuna.Trial) -> float:
    # 1. 移除 seed 搜索，固定随机性
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "device": device,
        "tree_method": "hist", # 如果有 GPU，改为 "gpu_hist"
        "device": "cuda",      # 如果有 GPU，取消注释
        "eta": trial.suggest_float("eta", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 20.0, log=True), # 改为 log 更合理
        "gamma": trial.suggest_float("gamma", 0.1, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 1.0),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 5.0),
        "scale_pos_weight": trial.suggest_float(
        "scale_pos_weight", 0.3 * neg_pos_ratio, 3.0 * neg_pos_ratio
        ),
        "seed": 42, # 固定种子
        # "n_jobs": -1,
    }

    fold_scores = []
    best_iterations = []
    
    # 2. 仅基于 Fold 进度进行 Pruning
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups)):
        
        # 显式使用 numpy 数组切片可能比 pandas iloc 更快（取决于具体数据结构）
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dvalid = xgb.DMatrix(X_va, label=y_va)

        # 3. 移除 XGBoostPruningCallback，避免 Step 冲突
        booster = xgb.train(
            params=params,
            dtrain=dtrain,
            num_boost_round=5000,
            evals=[(dtrain, "train"), (dvalid, "valid")],
            early_stopping_rounds=150,
            verbose_eval=False,
        )

        # 4. 直接使用 best_score 提升速度
        # 注意：booster.best_score 返回的是最后一次最佳迭代的分数
        # 如果 eval_metric 是 auc，这里就是 AUC
        best_score = booster.best_score 
        
        fold_scores.append(best_score)
        best_iterations.append(booster.best_iteration)

        # 清理内存
        del X_tr, X_va, y_tr, y_va, dtrain, dvalid, booster
        gc.collect()

        # 5. Fold 级别的 Pruning
        # 计算当前已完成 Fold 的平均分作为中间值
        current_mean_auc = np.mean(fold_scores)
        trial.report(current_mean_auc, step=fold_idx)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    mean_auc = np.mean(fold_scores)
    
    trial.set_user_attr("mean_best_iteration", int(np.mean(best_iterations)))
    
    return mean_auc

# 运行优化
study.optimize(objective, n_trials=100, gc_after_trial=True)


print("\n" + "="*30)
print("🎉 优化完成！结果如下：")
print("="*30)
# 输出最佳分数
print(f"🏆 最佳平均 AUC: {study.best_value:.6f}")

# 获取最佳迭代轮数 (如果在 objective 里通过 set_user_attr 保存了的话)
best_round = study.best_trial.user_attrs.get("mean_best_iteration")
print(f"🔄 最佳迭代轮数 (num_boost_round): {best_round}")

# ==========================================
# 3. 构建并打印最终可用的参数字典
# ==========================================
# 这里我们要把“固定参数”和“搜索到的最佳参数”合并
final_params = {
    # --- 固定参数 ---
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "device": "cuda",          # GPU
    "tree_method": "hist",     # GPU 模式
    "seed": 42,
    # --- 搜索到的最佳参数 ---
    **study.best_params
}

print("\n⚙️ 最终完整参数 (可直接复制):")
import pprint
pprint.pprint(final_params)

/home/yxcui/miniconda3/envs/nnunet_test/lib/python3.9/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2025-11-30 16:27:17,965] A new study created in memory with name: no-name-5568fdd0-518d-4592-87e0-6fbd3a97dbff
[I 2025-11-30 16:27:33,475] Trial 0 finished with value: 0.8101668291298803 and parameters: {'eta': 0.03574712922600244, 'max_depth': 9, 'min_child_weight': 8.960785365368121, 'gamma': 1.5751320499779735, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'colsample_bylevel': 0.6232334448672797, 'lambda': 2.9154431891537547, 'alpha': 0.2537815508265665, 'max_delta_step': 3.540362888980227, 'scale_pos_weight': 1.665008725501693}. Best is trial 0 with value: 0.8101668291298803.
[I 2025-11-30 16:27:48,360] Trial 1 finished with value: 0.7945109761964388 and parameters: {'eta': 0.2708160864249968, 'max_depth': 8, 'min_child_weight': 


🎉 优化完成！结果如下：
🏆 最佳平均 AUC: 0.816764
🔄 最佳迭代轮数 (num_boost_round): 262

⚙️ 最终完整参数 (可直接复制):
{'alpha': 8.89060752460763,
 'colsample_bylevel': 0.705591340606266,
 'colsample_bytree': 0.8570828860703237,
 'device': 'cuda',
 'eta': 0.03366081786187533,
 'eval_metric': 'auc',
 'gamma': 0.1390227680391509,
 'lambda': 0.3664212934115044,
 'max_delta_step': 3.021977328288438,
 'max_depth': 8,
 'min_child_weight': 18.05481185935681,
 'objective': 'binary:logistic',
 'scale_pos_weight': 3.5884467397465722,
 'seed': 42,
 'subsample': 0.9007638953963462,
 'tree_method': 'hist'}


### Catboost

In [7]:
import gc
import numpy as np
import pandas as pd
import optuna
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# 确保 groups 存在，sgkf 已定义
if 'groups' not in locals():
    groups = X_train.index 

# 重新计算正负比 (用于 scale_pos_weight)
neg_pos_ratio = (len(y_train) - y_train.sum()) / y_train.sum()

# 定义通用的 Study 参数
sampler = TPESampler(multivariate=True, n_startup_trials=15, seed=42)
pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

/home/yxcui/miniconda3/envs/nnunet_test/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/yxcui/miniconda3/envs/nnunet_test/lib/python3.9/site-packages/optuna/_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(


In [ ]:
def objective_cat(trial: optuna.Trial) -> float:
    params = {
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "task_type": "GPU", # CatBoost GPU 支持非常好
        "devices": "0",
        "verbose": 0,
        "random_seed": 42,
        
        # === 搜索空间 ===
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
        "border_count": 254, # GPU 上默认是 128，设为 254 精度更高
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5 * neg_pos_ratio, 2.0 * neg_pos_ratio),
    }

    fold_scores = []
    
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups)):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        train_pool = Pool(X_tr, y_tr)
        valid_pool = Pool(X_va, y_va)

        model = CatBoostClassifier(**params, iterations=5000)
        
        model.fit(
            train_pool,
            eval_set=valid_pool,
            early_stopping_rounds=150,
            verbose=0,
            use_best_model=True
        )

        # CatBoost 获取 best_score 有点麻烦，直接预测算 AUC 最稳
        preds = model.predict_proba(valid_pool)[:, 1]
        score = roc_auc_score(y_va, preds)
        fold_scores.append(score)
        
        del X_tr, X_va, y_tr, y_va, train_pool, valid_pool, model
        gc.collect()

        trial.report(np.mean(fold_scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_scores)

# 开始调优 CatBoost
print("🚀 开始 CatBoost 调优...")
study_cat = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study_cat.optimize(objective_cat, n_trials=100, gc_after_trial=True)
print(f"CatBoost 最佳 AUC: {study_cat.best_value}")

[I 2025-11-30 17:50:59,670] A new study created in memory with name: no-name-82773db0-072c-4b76-898a-8b7fc3047332


🚀 开始 CatBoost 调优...


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
[I 2025-11-30 17:51:21,610] Trial 0 finished with value: 0.7914365012650623 and parameters: {'learning_rate': 0.08012737503998542, 'depth': 4, 'l2_leaf_reg': 0.01474275315991467, 'random_strength': 0.029204338471814112, 'bagging_temperature': 0.45606998421703593, 'scale_pos_weight': 7.856196236768389}. Best is trial 0 with value: 0.7914365012650623.
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric per

### 为LightGBM做清洗列名+训练

In [9]:
import re
import pandas as pd

def clean_feature_names(df):
    """
    将列名中所有非(字母、数字、下划线)的字符替换为下划线
    解决 LightGBMError: Do not support special JSON characters in feature name
    """
    # 1. 获取原始列名
    original_cols = df.columns
    
    # 2. 使用正则替换：[^A-Za-z0-9_] 表示匹配所有 非字母数字下划线 的字符
    new_cols = [re.sub(r'[^A-Za-z0-9_]', '_', str(col)) for col in original_cols]
    
    # 3. 赋值回 DataFrame
    df.columns = new_cols
    
    return df

# === 执行清洗 ===
print("🧹 正在清洗列名...")
X_train = clean_feature_names(X_train)

# 如果你定义了 X_test，也要一起清洗，否则以后预测会报错列名不匹配
if 'X_test' in locals():
    X_test = clean_feature_names(X_test)

# === 验证清洗结果 ===
# 检查是否还有非法字符
bad_cols = [c for c in X_train.columns if re.search(r'[^A-Za-z0-9_]', str(c))]

if len(bad_cols) > 0:
    print(f"❌ 清洗失败！依然发现 {len(bad_cols)} 个非法列名:")
    print(bad_cols[:5])
else:
    print("✅ 清洗成功！所有列名均符合 LightGBM 要求。")
    print("示例列名:", list(X_train.columns[:5]))

🧹 正在清洗列名...
✅ 清洗成功！所有列名均符合 LightGBM 要求。
示例列名: ['seq_len', 'age__last', 'gender__last', 'ethnicity__last', 'Y90_Y99__last']


In [ ]:
# 开始训练
def objective_lgbm(trial: optuna.Trial) -> float:
    # LightGBM 特有的参数空间
    params = {
        "objective": "binary",
        "metric": "auc",
        "verbosity": -1,
        "boosting_type": "gbdt",
        
        # === GPU 设置 ===
        "device": "cpu", # 如果报错，请改为 "cpu"
        # "gpu_platform_id": 0,
        # "gpu_device_id": 0,
        
        # === 搜索空间 ===
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 256), # LGBM 最关键参数
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 7),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 0.5 * neg_pos_ratio, 2.0 * neg_pos_ratio),
        
        "seed": 42,
        "n_jobs": -1,
    }

    fold_scores = []
    
    for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups)):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]

        # LightGBM Dataset
        dtrain = lgb.Dataset(X_tr, label=y_tr)
        dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain)

        # 训练
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=5000,
            valid_sets=[dtrain, dvalid],
            valid_names=["train", "valid"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=150, verbose=False),
                lgb.log_evaluation(0) # 不打印日志
            ]
        )
        
        # 获取最佳分数 (AUC)
        # model.best_score['valid']['auc']
        best_score = model.best_score['valid']['auc']
        fold_scores.append(best_score)
        
        del X_tr, X_va, y_tr, y_va, dtrain, dvalid, model
        gc.collect()

        # Pruning
        trial.report(np.mean(fold_scores), step=fold_idx)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_scores)

# 开始调优 LightGBM
print("🚀 开始 LightGBM 调优...")
study_lgbm = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study_lgbm.optimize(objective_lgbm, n_trials=100, gc_after_trial=True)
print(f"LightGBM 最佳 AUC: {study_lgbm.best_value}")

[I 2025-11-30 19:52:48,247] A new study created in memory with name: no-name-75716d73-5c9f-40f1-ac7d-4a5ec6ada078


🧹 正在清洗列名...
✅ 清洗成功！所有列名均符合 LightGBM 要求。
示例列名: ['seq_len', 'age__last', 'gender__last', 'ethnicity__last', 'Y90_Y99__last']
🚀 开始 LightGBM 调优...


[I 2025-11-30 19:53:07,754] Trial 0 finished with value: 0.8071598927674964 and parameters: {'learning_rate': 0.03574712922600244, 'num_leaves': 245, 'max_depth': 10, 'min_child_samples': 62, 'subsample': 0.5780093202212182, 'subsample_freq': 2, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 2.9154431891537547, 'reg_lambda': 0.2537815508265665, 'scale_pos_weight': 7.314636756742065}. Best is trial 0 with value: 0.8071598927674964.
[I 2025-11-30 19:53:46,769] Trial 1 finished with value: 0.8115043988905863 and parameters: {'learning_rate': 0.010725209743171997, 'num_leaves': 249, 'max_depth': 11, 'min_child_samples': 25, 'subsample': 0.5909124836035503, 'subsample_freq': 2, 'colsample_bytree': 0.6521211214797689, 'reg_alpha': 0.12561043700013558, 'reg_lambda': 0.05342937261279776, 'scale_pos_weight': 4.3868078498037075}. Best is trial 1 with value: 0.8115043988905863.
[I 2025-11-30 19:53:58,597] Trial 2 finished with value: 0.8095414534709369 and parameters: {'learning_rate': 0.08

LightGBM 最佳 AUC: 0.8166491535386863


### 结合三个模型+预测testing set，输出csv

In [13]:
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
import gc
import re
from pathlib import Path

# ==========================================
# 0. 配置路径与辅助函数
# ==========================================
OUTPUT_PATH = Path("/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/3model_submission.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True) # 确保目录存在

def clean_feature_name(name):
    """清洗特征名，防止 LightGBM 报错"""
    return re.sub(r'[^A-Za-z0-9_]', '_', str(name))

# 统一清洗所有数据的列名，避免后续麻烦
print("🧹 Cleaning feature names...")
X_train.columns = [clean_feature_name(c) for c in X_train.columns]
X_test.columns = [clean_feature_name(c) for c in X_test.columns]
feature_names = list(X_train.columns)

# ==========================================
# 1. 准备最佳参数
# ==========================================
# final_params_xgb = {
#     'objective': 'binary:logistic',
#     'eval_metric': 'auc',
#     'tree_method': 'hist',
#     'device': 'cuda', 
#     'seed': 42,
#     'eta': 0.03366081786187533,
#     'max_depth': 8,
#     'min_child_weight': 18.05481185935681,
#     'gamma': 0.1390227680391509,
#     'subsample': 0.9007638953963462,
#     'colsample_bytree': 0.8570828860703237,
#     'colsample_bylevel': 0.705591340606266,
#     'alpha': 8.89060752460763,
#     'lambda': 0.3664212934115044,
#     'max_delta_step': 3.021977328288438,
#     'scale_pos_weight': 3.5884467397465722,
# }
final_params_xgb = {
    # === 核心训练任务参数 ===
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',  # 确保环境支持 GPU
    'seed': 4,        # 这里保持固定种子以便复现，也可改为调参时的 seed: 4
    
    # === 搜索到的最佳超参数 ===
    'eta': 0.01052553111520763,
    'max_depth': 7,
    'min_child_weight': 3.6087229241854986,
    'gamma': 2.6462251747249104,
    'subsample': 0.9049042481817137,
    'colsample_bytree': 0.9331777936528035,
    'colsample_bylevel': 0.962007606168361,
    
    # === 正则化参数 ===
    'alpha': 0.015938815227799684,
    'lambda': 0.6564364262394898,
    
    # === 不平衡处理参数 ===
    'max_delta_step': 1.700596084807252,
    'scale_pos_weight': 1.4805937749571343,
}

final_params_lgb = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'device': 'gpu',
    'seed': 42,
    'learning_rate': 0.010536814123395107,
    'num_leaves': 161,
    'max_depth': 11,
    'min_child_samples': 90,
    'subsample': 0.8970229931944054,
    'subsample_freq': 4,
    'colsample_bytree': 0.6782012937276678,
    'reg_alpha': 0.001827144435952068,
    'reg_lambda': 0.003362813910357317,
    'scale_pos_weight': 2.4584247976591955,
}

final_params_cat = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 42,
    'verbose': 0,
    'metric_period': 5,
    'border_count': 254,
    'learning_rate': 0.017728997012793797, 
    'depth': 9,
    'l2_leaf_reg': 9.741243264206538,
    'random_strength': 0.025245857049830272,
    'bagging_temperature': 0.5122126737008887,
    'scale_pos_weight': 4.692312693040222,
    'bootstrap_type': 'Bayesian',
}

# ==========================================
# 2. 初始化存储数组
# ==========================================
# A. 测试集预测结果 (用于提交)
test_pred_xgb = np.zeros(len(X_test))
test_pred_lgb = np.zeros(len(X_test))
test_pred_cat = np.zeros(len(X_test))

# B. 验证集 OOF (Out-Of-Fold) 预测结果 (用于计算 AUC/AUPRC)
oof_pred_xgb = np.zeros(len(X_train))
oof_pred_lgb = np.zeros(len(X_train))
oof_pred_cat = np.zeros(len(X_train))

# ==========================================
# 3. 5-Fold 循环推理
# ==========================================
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

# 确保 groups 存在
if 'groups' not in locals():
    groups = X_train.index 
    print("⚠️ Warning: 'groups' not found, using X_train.index.")

print("🚀 开始 5-Fold 集成推理...")

for fold_idx, (train_idx, valid_idx) in enumerate(sgkf.split(X_train, y_train, groups=groups)):
    print(f"\n--- Fold {fold_idx + 1} / 5 ---")
    
    # 切分数据
    X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[valid_idx]
    
    # ---------------------------
    # Model 1: XGBoost
    # ---------------------------
    dtrain = xgb.DMatrix(X_tr, label=y_tr, feature_names=feature_names)
    dvalid = xgb.DMatrix(X_va, label=y_va, feature_names=feature_names)
    dtest  = xgb.DMatrix(X_test, feature_names=feature_names)
    
    model_xgb = xgb.train(
        final_params_xgb, 
        dtrain, 
        num_boost_round=3000, 
        evals=[(dvalid, "valid")], 
        early_stopping_rounds=100, 
        verbose_eval=False
    )
    # 预测 OOF (用于验证)
    oof_pred_xgb[valid_idx] = model_xgb.predict(dvalid, iteration_range=(0, model_xgb.best_iteration + 1))
    # 预测 Test (用于提交)
    test_pred_xgb += model_xgb.predict(dtest, iteration_range=(0, model_xgb.best_iteration + 1)) / 5
    
    # ---------------------------
    # Model 2: LightGBM
    # ---------------------------
    # 注意：列名已在开头清洗过，这里直接用
    dtrain_lgb = lgb.Dataset(X_tr, label=y_tr, feature_name=feature_names)
    dvalid_lgb = lgb.Dataset(X_va, label=y_va, reference=dtrain_lgb, feature_name=feature_names)
    
    model_lgb = lgb.train(
        final_params_lgb, 
        dtrain_lgb, 
        num_boost_round=5000, 
        valid_sets=[dvalid_lgb], 
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    # LightGBM predict 直接接受 DataFrame
    oof_pred_lgb[valid_idx] = model_lgb.predict(X_va, num_iteration=model_lgb.best_iteration)
    test_pred_lgb += model_lgb.predict(X_test, num_iteration=model_lgb.best_iteration) / 5
    
    # ---------------------------
    # Model 3: CatBoost
    # ---------------------------
    train_pool = Pool(X_tr, y_tr, feature_names=feature_names)
    valid_pool = Pool(X_va, y_va, feature_names=feature_names)
    test_pool  = Pool(X_test, feature_names=feature_names)
    
    model_cat = CatBoostClassifier(**final_params_cat, iterations=5000)
    model_cat.fit(
        train_pool, 
        eval_set=valid_pool, 
        early_stopping_rounds=100, 
        verbose=0, 
        use_best_model=True
    )
    # CatBoost predict_proba 返回 (N, 2)，取第二列
    oof_pred_cat[valid_idx] = model_cat.predict_proba(valid_pool)[:, 1]
    test_pred_cat += model_cat.predict_proba(test_pool)[:, 1] / 5
    
    # 内存清理
    del X_tr, X_va, y_tr, y_va, dtrain, dvalid, dtest, dtrain_lgb, dvalid_lgb, train_pool, valid_pool, test_pool
    del model_xgb, model_lgb, model_cat
    gc.collect()

print("\n✅ 5-Fold 推理完成！")

# ==========================================
# 4. 计算验证集指标 (CV Score)
# ==========================================
print("\n📊 Calculating CV Scores (AUC & AUPRC)...")

# 计算单模型指标
xgb_auc = roc_auc_score(y_train, oof_pred_xgb)
lgb_auc = roc_auc_score(y_train, oof_pred_lgb)
cat_auc = roc_auc_score(y_train, oof_pred_cat)

print(f"XGBoost CV AUC : {xgb_auc:.6f}")
print(f"LightGBM CV AUC: {lgb_auc:.6f}")
print(f"CatBoost CV AUC: {cat_auc:.6f}")

# --- Rank Averaging on OOF (验证融合效果) ---
rank_oof_xgb = rankdata(oof_pred_xgb) / len(oof_pred_xgb)
rank_oof_lgb = rankdata(oof_pred_lgb) / len(oof_pred_lgb)
rank_oof_cat = rankdata(oof_pred_cat) / len(oof_pred_cat)

oof_rank_pred = (rank_oof_xgb + rank_oof_lgb + rank_oof_cat) / 3

ensemble_auc = roc_auc_score(y_train, oof_rank_pred)
ensemble_auprc = average_precision_score(y_train, oof_rank_pred)

print("-" * 30)
print(f"🏆 Ensemble CV AUC   : {ensemble_auc:.6f}")
print(f"🏆 Ensemble CV AUPRC : {ensemble_auprc:.6f}")
print("-" * 30)

# ==========================================
# 5. 生成提交文件 (Test Set)
# ==========================================
print("⚖️ Applying Rank Averaging on Test Set...")

rank_test_xgb = rankdata(test_pred_xgb) / len(test_pred_xgb)
rank_test_lgb = rankdata(test_pred_lgb) / len(test_pred_lgb)
rank_test_cat = rankdata(test_pred_cat) / len(test_pred_cat)

final_rank_pred = (rank_test_xgb + rank_test_lgb + rank_test_cat) / 3

submission_ids = X_test.index 
submission = pd.DataFrame({
    "id": submission_ids, 
    "readmitted_within_30days": final_rank_pred
})

submission.to_csv(OUTPUT_PATH, index=False)

print(f"🎉 结果已保存至绝对路径: {OUTPUT_PATH}")
print(submission.head())

🧹 Cleaning feature names...
🚀 开始 5-Fold 集成推理...

--- Fold 1 / 5 ---

--- Fold 2 / 5 ---

--- Fold 3 / 5 ---

--- Fold 4 / 5 ---

--- Fold 5 / 5 ---

✅ 5-Fold 推理完成！

📊 Calculating CV Scores (AUC & AUPRC)...
XGBoost CV AUC : 0.814363
LightGBM CV AUC: 0.813719
CatBoost CV AUC: 0.804730
------------------------------
🏆 Ensemble CV AUC   : 0.814903
🏆 Ensemble CV AUPRC : 0.648628
------------------------------
⚖️ Applying Rank Averaging on Test Set...
🎉 结果已保存至绝对路径: /home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/3model_submission.csv
                  id  readmitted_within_30days
0  19661325_29884966                  0.649155
1  16300198_24781018                  0.086100
2  16706302_24530345                  0.983947
3  13205882_26134830                  0.711054
4  14671276_21169914                  0.697677


### 其他 （用于测试）

In [11]:
# ================================================================
# Script 1: Train on (X_train, y_train), evaluate on (X_val, y_val)
# ================================================================
# 需求：
#   - 你已经在当前 Python 会话中准备好以下对象：
#       X_train, y_train, X_val, y_val  （全部为 pandas 对象，特征列齐全一致）
#   - best_params 字典可在此脚本中自定义或使用你已有的调参结果
# ================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

# ------------------------------
# 路径配置（如不需要可移除）
# ------------------------------
ARTIFACT_DIR = Path("/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

HOLDOUT_MODEL_PATH = ARTIFACT_DIR / "xgb_holdout_model.json"
SUMMARY_PATH = ARTIFACT_DIR / "xgb_holdout_summary.json"
FEATURE_NAMES_PATH = ARTIFACT_DIR / "feature_names.json"
BEST_PARAMS_PATH = ARTIFACT_DIR / "xgb_best_params.json"

# ------------------------------
# 你的调参结果
# ------------------------------
best_params: dict[str, float | int | str] = {
    "objective": "binary:logistic",
    "eval_metric": ["auc", "aucpr"],
    "tree_method": "hist",
    #'learning_rate': 0.012607804208705605, 'max_depth': 6, 'min_child_weight': 1.5441944900308036, 'subsample': 0.6474895359316174, 'colsample_bytree': 0.6653689521216923, 'colsample_bylevel': 0.8409173323719815, 'gamma': 0.199722686457338, 'lambda': 0.3310867254800588, 'alpha': 0.00938718543561836, 'scale_pos_weight': 8.477948815127707, 'max_delta_step': 4.3745787481587834,
    #"seed": 42,
    #'eta': 0.026617192493058012, 'max_depth': 7, 'min_child_weight': 10.980621078783743, 'gamma': 0.25807415669615313, 'subsample': 0.9435064665118181, 'colsample_bytree': 0.553215083593131, 'colsample_bylevel': 0.7347203452424912, 'lambda': 0.3037867477831362, 'alpha': 0.07651806525510074, 'max_delta_step': 1.3852581338737902, 'scale_pos_weight': 1.8180089854898651, 'seed': 1,
    
    #四百多，目前最好
    'eta': 0.01052553111520763, 'max_depth': 7, 'min_child_weight': 3.6087229241854986, 'gamma': 2.6462251747249104, 'subsample': 0.9049042481817137, 'colsample_bytree': 0.9331777936528035, 'colsample_bylevel': 0.962007606168361, 'lambda': 0.6564364262394898, 'alpha': 0.015938815227799684, 'max_delta_step': 1.700596084807252, 'scale_pos_weight': 1.4805937749571343, 'seed': 4,

    #760
    #'eta': 0.023063389951490133, 'max_depth': 7, 'min_child_weight': 9.051055082081746, 'gamma': 0.19469215076455676, 'subsample': 0.7941190124159108, 'colsample_bytree': 0.9310768318170357, 'colsample_bylevel': 0.782839478253501, 'lambda': 0.32174577181733277, 'alpha': 0.0128061602620061, 'max_delta_step': 1.6002553105236, 'scale_pos_weight': 3.265727264059701, 'seed': 4,
    #这个是用了visit_count的，auc=0.89,but不能用
    # 'eta': 0.030655678231142226, 'max_depth': 7, 'min_child_weight': 5.753357031542597, 'gamma': 1.3167289277154512, 'subsample': 0.8169387709767664, 'colsample_bytree': 0.5860026442370233, 'colsample_bylevel': 0.7091712037975286, 'lambda': 0.4946130323361874, 'alpha': 2.650989539871699, 'max_delta_step': 0.4128988178135878, 'scale_pos_weight': 8.374416649237142, 'seed': 7,
}
# 写入 JSON 文件
BEST_PARAMS_PATH.write_text(json.dumps(best_params, indent=2))

# 常量指向超参字典本身
BEST_PARAMS = best_params

NUM_BOOST_ROUND = 4000
EARLY_STOPPING_ROUNDS = 200
VERBOSE_EVAL = 100

# ------------------------------
# 构建 DMatrix 工具
# ------------------------------
def build_dmatrix(
    X: pd.DataFrame,
    y: pd.Series | None = None,
    *,
    feature_order: list[str] | None = None,
) -> xgb.DMatrix:
    if feature_order is not None:
        missing_cols = [col for col in feature_order if col not in X.columns]
        if missing_cols:
            X = X.copy()
            for col in missing_cols:
                X[col] = 0.0
        X = X.reindex(columns=feature_order, fill_value=0.0)

    X_np = X.astype(np.float32, copy=False)
    feature_names = [str(col) for col in X.columns]

    duplicates = pd.Series(feature_names).value_counts()
    if (duplicates > 1).any():
        dup_list = duplicates[duplicates > 1].index.tolist()
        raise ValueError(f"Duplicate feature names detected: {dup_list}")

    if y is None:
        return xgb.DMatrix(data=X_np, feature_names=feature_names)

    labels = y.astype(np.float32, copy=False).to_numpy()
    return xgb.DMatrix(data=X_np, label=labels, feature_names=feature_names)

# ------------------------------
# 训练 + 验证
# ------------------------------
assert list(X_train.columns) == list(X_val.columns), "train/val 特征顺序不一致"

feature_order = [str(col) for col in X_train.columns]
dtrain = build_dmatrix(X_train, y_train, feature_order=feature_order)
dval = build_dmatrix(X_val, y_val, feature_order=feature_order)

early_stop = xgb.callback.EarlyStopping(
    rounds=EARLY_STOPPING_ROUNDS,
    metric_name="auc",
    maximize=True,
    save_best=True,
)

booster = xgb.train(
    params=best_params,
    dtrain=dtrain,
    num_boost_round=NUM_BOOST_ROUND,
    evals=[(dtrain, "train"), (dval, "holdout")],
    callbacks=[early_stop],
    verbose_eval=VERBOSE_EVAL,
)

best_iteration = booster.best_iteration
print(f"\nBest iteration (holdout): {best_iteration}")
booster.save_model(HOLDOUT_MODEL_PATH)
FEATURE_NAMES_PATH.write_text(json.dumps(dtrain.feature_names))

# ------------------------------
# 验证集评估
# ------------------------------
val_pred = booster.predict(dval, iteration_range=(0, best_iteration + 1))
val_auc = roc_auc_score(y_val, val_pred)
val_auprc = average_precision_score(y_val, val_pred)

fpr, tpr, roc_thresholds = roc_curve(y_val, val_pred)
youden_idx = np.argmax(tpr - fpr)
best_threshold_roc = float(roc_thresholds[youden_idx])

precision, recall, pr_thresholds = precision_recall_curve(y_val, val_pred)
f_scores = 2 * precision * recall / (precision + recall + 1e-8)
best_f_idx = int(np.argmax(f_scores))
best_threshold_f1 = float(pr_thresholds[best_f_idx])

print(f"Hold-out AUC   : {val_auc:.6f}")
print(f"Hold-out AUPRC : {val_auprc:.6f}")
print(f"Best ROC threshold (Youden): {best_threshold_roc:.4f}")
print(
    f"Best F1 threshold          : {best_threshold_f1:.4f} "
    f"(Precision={precision[best_f_idx]:.3f}, Recall={recall[best_f_idx]:.3f})"
)

summary = {
    "best_iteration": int(best_iteration),
    "holdout_auc": float(val_auc),
    "holdout_auprc": float(val_auprc),
    "best_threshold_roc": best_threshold_roc,
    "best_threshold_f1": best_threshold_f1,
    "precision_at_best_f1": float(precision[best_f_idx]),
    "recall_at_best_f1": float(recall[best_f_idx]),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2))

print(f"\nSaved hold-out model -> {HOLDOUT_MODEL_PATH}")
print(f"Saved summary        -> {SUMMARY_PATH}")
print(f"Saved feature names  -> {FEATURE_NAMES_PATH}")
print(f"Saved best params    -> {BEST_PARAMS_PATH}")

[0]	train-auc:0.79270	train-aucpr:0.51470	holdout-auc:0.72750	holdout-aucpr:0.41254
[100]	train-auc:0.88902	train-aucpr:0.76861	holdout-auc:0.77790	holdout-aucpr:0.59325
[200]	train-auc:0.91609	train-aucpr:0.81664	holdout-auc:0.78306	holdout-aucpr:0.60660
[300]	train-auc:0.93705	train-aucpr:0.85374	holdout-auc:0.78508	holdout-aucpr:0.61215
[400]	train-auc:0.95563	train-aucpr:0.88725	holdout-auc:0.78667	holdout-aucpr:0.61580
[500]	train-auc:0.96802	train-aucpr:0.91227	holdout-auc:0.78750	holdout-aucpr:0.61741
[600]	train-auc:0.97772	train-aucpr:0.93431	holdout-auc:0.78758	holdout-aucpr:0.61900
[700]	train-auc:0.98480	train-aucpr:0.95245	holdout-auc:0.78799	holdout-aucpr:0.62036
[800]	train-auc:0.98767	train-aucpr:0.96040	holdout-auc:0.78726	holdout-aucpr:0.62014
[855]	train-auc:0.98849	train-aucpr:0.96280	holdout-auc:0.78711	holdout-aucpr:0.62031

Best iteration (holdout): 655
Hold-out AUC   : 0.788981
Hold-out AUPRC : 0.620644
Best ROC threshold (Youden): 0.2318
Best F1 threshold      

In [9]:
# ================================================================
# Script 2: Merge train+val, retrain final model, predict on X_test
# ================================================================
# 需求：
#   - 当前环境中已有：
#       X_train, y_train, X_val, y_val, X_test, sample_submission
#       best_iteration（由 Script 1 产生）或自行指定
#   - sample_submission 必须包含 "id" 列和目标列（或待填充列）
# ================================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb

# ------------------------------
# 路径配置
# ------------------------------
ARTIFACT_DIR = Path("/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FULL_MODEL_PATH = ARTIFACT_DIR / "xgb_full_model.json"
FEATURE_NAMES_PATH = ARTIFACT_DIR / "feature_names.json"
SUBMISSION_PATH = ARTIFACT_DIR / "submission.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)
sample_submission["id"] = sample_submission["id"].astype(str)
sample_submission = sample_submission.set_index("id").reindex(test_ids).reset_index()

# ------------------------------
# 获取 best_params 与 best_iteration
# ------------------------------
best_params = json.loads((ARTIFACT_DIR / "xgb_best_params.json").read_text())
holdout_summary = json.loads((ARTIFACT_DIR / "xgb_holdout_summary.json").read_text())
best_iteration = holdout_summary["best_iteration"]

# 如需手动指定轮数，可替换上面两行：
# best_params = {...}
# best_iteration = 1234

# ------------------------------
# 合并 train + val
# ------------------------------
assert list(X_train.columns) == list(X_val.columns) == list(X_test.columns), "特征列不一致"

X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

# ------------------------------
# 构建 DMatrix
# ------------------------------
def build_dmatrix(
    X: pd.DataFrame,
    y: pd.Series | None = None,
    *,
    feature_order: list[str] | None = None,
) -> xgb.DMatrix:
    if feature_order is not None:
        missing_cols = [col for col in feature_order if col not in X.columns]
        if missing_cols:
            X = X.copy()
            for col in missing_cols:
                X[col] = 0.0
        X = X.reindex(columns=feature_order, fill_value=0.0)

    X_np = X.astype(np.float32, copy=False)
    feature_names = [str(col) for col in X.columns]

    duplicates = pd.Series(feature_names).value_counts()
    if (duplicates > 1).any():
        dup_list = duplicates[duplicates > 1].index.tolist()
        raise ValueError(f"Duplicate feature names detected: {dup_list}")

    if y is None:
        return xgb.DMatrix(data=X_np, feature_names=feature_names)

    labels = y.astype(np.float32, copy=False).to_numpy()
    return xgb.DMatrix(data=X_np, label=labels, feature_names=feature_names)

feature_order = [str(col) for col in X_full.columns]
dfull = build_dmatrix(X_full, y_full, feature_order=feature_order)

# ------------------------------
# 训练最终模型
# ------------------------------
booster_full = xgb.train(
    params=best_params,
    dtrain=dfull,
    num_boost_round=best_iteration + 1,
    verbose_eval=False,
)
booster_full.save_model(FULL_MODEL_PATH)
FEATURE_NAMES_PATH.write_text(json.dumps(dfull.feature_names))

# ------------------------------
# 预测测试集并生成提交
# ------------------------------
X_test_ordered = X_test.copy()
if "id" in X_test_ordered.columns:
    X_test_ordered = X_test_ordered.set_index("id")
X_test_ordered.index = X_test_ordered.index.astype(str)

# 确保 sample_submission 可用
sample_submission = sample_submission.copy()
sample_submission["id"] = sample_submission["id"].astype(str)

test_ids = sample_submission["id"].tolist()
X_test_ordered = X_test_ordered.reindex(test_ids)
missing_ids = [idx for idx in test_ids if idx not in X_test_ordered.index]
if missing_ids:
    raise ValueError(f"X_test 缺少 {len(missing_ids)} 个样本，例如: {missing_ids[:5]}")

dtest = build_dmatrix(X_test_ordered, feature_order=dfull.feature_names)
test_pred = booster_full.predict(dtest)

target_column = sample_submission.columns[-1]
sample_submission[target_column] = test_pred.astype(np.float32)
sample_submission.to_csv(SUBMISSION_PATH, index=False)


print(f"Saved full model        -> {FULL_MODEL_PATH}")
print(f"Saved feature names     -> {FEATURE_NAMES_PATH}")
print(f"Saved Kaggle submission -> {SUBMISSION_PATH}")

Saved full model        -> /home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/artifacts/xgb_full_model.json
Saved feature names     -> /home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/artifacts/feature_names.json
Saved Kaggle submission -> /home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/hys_code/artifacts/submission.csv


## Track 2

In [12]:
from __future__ import annotations
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, List

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import xgboost as xgb

import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
device = "cuda" if torch.cuda.is_available() else "cpu"

In [2]:
DATA_DIR = Path("/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project")

# 结构化表格（每行一个住院样本，含标签/静态信息）
train_df = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/train.csv")   # 至少包含 id, label, 性别, 年龄, ...
valid_df  = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/valid.csv")
test_df = pd.read_csv(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/test.csv")

#print(train_df.info())
#print(test_df.info())

# 时序特征字典：key 为 id，value 为 (n_days, 171) 的 numpy 数组
ehr_data = np.load(DATA_DIR / "/home/yxcui/HKUcourse/STAT3612/group_project/2025-fall-stat-3612-group-project/ehr_preprocessed_seq_by_day_cat_embedding.pkl", allow_pickle=True)

feat_dict = ehr_data["feat_dict"]
feature_cols: List[str] = ehr_data["feature_cols"]

print(type(ehr_data))
print(type(feat_dict), len(feat_dict))
print(list(feat_dict.keys())[:5])        # 只看前几个病人 ID
print(feature_cols[:10])                 # 只看前 10 个特征名
print(feat_dict[next(iter(feat_dict))].shape)  # 查看单个样本的数组尺寸

<class 'dict'>
<class 'dict'> 14532
['10869829_25238191', '17910612_22301530', '16026764_21404901', '12347278_29852086', '18463717_24608289']
['age', 'gender', 'ethnicity', 'Y90-Y99', 'G30-G32', 'O85-O92', 'C60-C63', 'F40-F48', 'M80-M85', 'R00-R09']
(13, 171)


In [4]:
demo_cols = ehr_data["demo_cols"]  # 如果有不同列名，按实际修改

icd_cols = ehr_data["icd_cols"]
lab_cols = ehr_data["lab_cols"]
med_cols = ehr_data["med_cols"]

demo_idxs = [feature_cols.index(c) for c in demo_cols]
icd_idxs  = [feature_cols.index(c) for c in icd_cols]
lab_idxs  = [feature_cols.index(c) for c in lab_cols]
med_idxs  = [feature_cols.index(c) for c in med_cols]

print(len(demo_idxs), len(icd_idxs), len(lab_idxs), len(med_idxs))
print(icd_cols, lab_cols, med_cols)

3 91 36 41
['Y90-Y99', 'G30-G32', 'O85-O92', 'C60-C63', 'F40-F48', 'M80-M85', 'R00-R09', 'J90-J94', 'A00-A09', 'E00-E07', 'F01-F09', 'F30-F39', 'H30-H36', 'D60-D64', 'N00-N08', 'F60-F69', 'I80-I89', 'I95-I99', 'N30-N39', 'K55-K64', 'F50-F59', 'R40-R46', 'J60-J70', 'N20-N23', 'I30-I52', 'R50-R69', 'B25-B34', 'C00-C14', 'D65-D69', 'C73-C75', 'G35-G37', 'E70-E88', 'K20-K31', 'C30-C39', 'I70-I79', 'M60-M63', 'A20-A28', 'N10-N16', 'E89-E89', 'R70-R79', 'C7B-C7B', 'M50-M54', 'A30-A49', 'F20-F29', 'G89-G99', 'R30-R39', 'J30-J39', 'N25-N29', 'Q65-Q79', 'G20-G26', 'B20-B20', 'K65-K68', 'R10-R19', 'E65-E68', 'B65-B83', 'E40-E46', 'F70-F79', 'N17-N19', 'J09-J18', 'J40-J47', 'C15-C26', 'L80-L99', 'B50-B64', 'O60-O77', 'C7A-C7A', 'B85-B89', 'E50-E64', 'K00-K14', 'R20-R23', 'J00-J06', 'N60-N65', 'D37-D48', 'K35-K38', 'G00-G09', 'M05-M14', 'I10-I16', 'B35-B49', 'K40-K46', 'K70-K77', 'Q00-Q07', 'E20-E35', 'J20-J22', 'A80-A89', 'B00-B09', 'J80-J84', 'G60-G65', 'D3A-D3A', 'G10-G14', 'B90-B94', 'N40-N53'

In [5]:
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------
# Feature engineering utilities
# ---------------------------------------------------------------------

def _compute_slope(series: np.ndarray) -> float:
    """最小二乘拟合趋势线斜率（按时间顺序的变化率）。"""
    mask = ~np.isnan(series)
    if mask.sum() < 2:
        return 0.0

    y = series[mask].astype(float, copy=False)
    x = np.flatnonzero(mask).astype(float)

    if np.allclose(y, y.mean()):
        return 0.0

    return float(np.polyfit(x, y, 1)[0])


def summarize_static(series: np.ndarray, prefix: str, seq_len: int | None = None) -> dict[str, float]:
    """静态/缓慢变化特征：仅保留最后一个非缺失值。"""
    mask = ~np.isnan(series)
    if not mask.any():
        return {f"{prefix}__last": np.nan}

    last_value = float(series[mask][-1])
    return {f"{prefix}__last": last_value}


def summarize_lab(series: np.ndarray, prefix: str, seq_len: int | None = None) -> dict[str, float]:
    """
    动态实验室检查：计算均值、标准差、最值、首末值、变化量、斜率、观测次数。
    """
    mask = ~np.isnan(series)
    if not mask.any():
        return {
            f"{prefix}__mean": np.nan,
            f"{prefix}__std": np.nan,
            f"{prefix}__min": np.nan,
            f"{prefix}__max": np.nan,
            f"{prefix}__first": np.nan,
            f"{prefix}__last": np.nan,
            f"{prefix}__delta": np.nan,
            f"{prefix}__slope": 0.0,
            f"{prefix}__slope*days": 0.0,
        }

    values = series[mask].astype(float, copy=False)
    first_val = float(values[0])
    last_val = float(values[-1])
    slope = _compute_slope(series.astype(float, copy=False))
    effective_len = seq_len if seq_len is not None else mask.sum()

    return {
        f"{prefix}__mean": float(values.mean()),
        f"{prefix}__std": float(values.std(ddof=0)),
        f"{prefix}__min": float(values.min()),
        f"{prefix}__max": float(values.max()),
        f"{prefix}__first": first_val,
        f"{prefix}__last": last_val,
        f"{prefix}__delta": last_val - first_val,
        f"{prefix}__slope": slope,
        f"{prefix}__slope*days": slope * float(effective_len),
    }


def extract_patient_id(pid: str) -> str:
    """
    默认用第一个分隔符（优先 '_'，其次 '-', '__', '::'）前的部分作为 patient_id。
    若无分隔符，则直接返回原始 pid。
    """
    if not isinstance(pid, str):
        return str(pid)

    for sep in ("_", "-", "__", "::"):
        if sep in pid:
            return pid.split(sep, 1)[0]
    return pid


# ---------------------------------------------------------------------
# Group configuration
# ---------------------------------------------------------------------
# 预先准备好的列名与索引列表（需与 feat_dict 对应）
# demo_cols, demo_idxs, lab_cols, lab_idxs, icd_cols, icd_idxs, med_cols, med_idxs
# 在此假定这些列表以及 feat_dict 已经定义好。

group_meta = {
    "demo": {"cols": demo_cols, "idxs": demo_idxs, "agg": summarize_static},
    "icd":  {"cols": icd_cols,  "idxs": icd_idxs,  "agg": summarize_static},
    "med":  {"cols": med_cols,  "idxs": med_idxs,  "agg": summarize_static},
    "lab":  {"cols": lab_cols,  "idxs": lab_idxs,  "agg": summarize_lab},
}

# ---------------------------------------------------------------------
# Patient-level feature collapse
# ---------------------------------------------------------------------

patient_rows: list[dict[str, float]] = []

for pid, arr in feat_dict.items():
    arr = np.asarray(arr, dtype=np.float32)
    seq_len = arr.shape[0]

    row: dict[str, float] = {
        "pid": pid,
        "seq_len": float(seq_len),
    }

    for group_name, meta in group_meta.items():
        cols = meta["cols"]
        idxs = meta["idxs"]
        agg_fn = meta["agg"]

        if not cols:
            continue

        if len(cols) != len(idxs):
            raise ValueError(f"{group_name}: cols 与 idxs 长度不一致")

        block = arr[:, idxs]  # shape: (seq_len, num_features)

        for col_name, series in zip(cols, block.T):
            row.update(agg_fn(series, prefix=col_name, seq_len=seq_len))

    patient_rows.append(row)

feature_df = (
    pd.DataFrame(patient_rows)
      .set_index("pid")
      .astype(np.float32)
      .sort_index()
)

# 检查结果
print(feature_df.shape)
print(feature_df.head())

(14532, 460)
                   seq_len  age__last  gender__last  ethnicity__last  \
pid                                                                    
10001884_26184834     14.0       77.0           0.0              2.0   
10002013_23581541      6.0       57.0           0.0              6.0   
10002428_23473524     12.0       81.0           0.0              6.0   
10002428_28662225     18.0       81.0           0.0              6.0   
10003019_20962108      9.0       74.0           1.0              6.0   

                   Y90-Y99__last  G30-G32__last  O85-O92__last  C60-C63__last  \
pid                                                                             
10001884_26184834            0.0            0.0            0.0            0.0   
10002013_23581541            0.0            0.0            0.0            0.0   
10002428_23473524            0.0            0.0            0.0            0.0   
10002428_28662225            0.0            0.0            0.0            0.0

In [6]:
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# -------------------------------------------------
# 1. 读入 notes.csv，构造 pid
# -------------------------------------------------
notes_df = pd.read_csv(DATA_DIR / "notes.csv")
notes_df["pid"] = (
    notes_df["subject_id"].astype(str) + "_" +
    notes_df["hadm_id"].astype(str)
)
notes_df["text"] = notes_df["text"].fillna("")

# -------------------------------------------------
# 2. 将文本转换为 TF‑IDF 向量
# -------------------------------------------------
vectorizer = TfidfVectorizer(
    min_df=5,          # 至少出现在 5 个样本中的词才保留
    max_df=0.8,        # 出现频率 >80% 的词视为过于常见，剔除
    max_features=512,  # 限制特征数，避免矩阵太大 512
    ngram_range=(1, 2),
    norm="l2"
)

note_matrix = vectorizer.fit_transform(notes_df["text"])

note_feat_df = pd.DataFrame.sparse.from_spmatrix(
    note_matrix,
    index=notes_df["pid"],
    columns=[f"note_tfidf_{term}" for term in vectorizer.get_feature_names_out()]
)

# -------------------------------------------------
# 3. 以 pid 聚合（同一个病人可能多条 note）
#    这里取平均，也可换成 max / sum 等策略
# -------------------------------------------------
note_feat_df = note_feat_df.groupby(level=0).mean()

In [9]:
note_feat_df = note_feat_df.reindex(feature_df.index).fillna(0)
combined_df = feature_df.join(note_feat_df, how="left").fillna(0)

print(combined_df.shape)
combined_df.head()

(14532, 465)


,seq_len,age__last,gender__last,ethnicity__last,Y90-Y99__last,G30-G32__last,O85-O92__last,C60-C63__last,F40-F48__last,M80-M85__last,...,pCO2 Blood__first,pCO2 Blood__last,pCO2 Blood__delta,pCO2 Blood__slope,pCO2 Blood__slope*days,note_tfidf_he,note_tfidf_her,note_tfidf_his,note_tfidf_she,note_tfidf_tablet
pid,,,,,,,,,,,,,,,,,,,,,
10001884_26184834,14.0,77.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.032967,0.461538,0,0.651185,0,0.758919,0
10002013_23581541,6.0,57.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.142857,0.857143,0,0.203263,0,0.335597,0.919815
10002428_23473524,12.0,81.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,1.0,0.087413,1.048951,0,0.61978,0,0.777696,0.105174
10002428_28662225,18.0,81.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,-0.017544,-0.315789,0,0.198629,0,0.934645,0.294933
10003019_20962108,9.0,74.0,1.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.050000,0.450000,0.90272,0,0.392655,0,0.175838


In [10]:
# === 1. 准备 train / valid 的标签 ======================================
train_labels = (
    train_df.loc[:, ["id", "readmitted_within_30days"]]
            .rename(columns={"readmitted_within_30days": "label"})
            .assign(split="train")
)

valid_labels = (
    valid_df.loc[:, ["id", "readmitted_within_30days"]]
            .rename(columns={"readmitted_within_30days": "label"})
            .assign(split="valid")
)

label_df = pd.concat([train_labels, valid_labels], ignore_index=True)
label_df["id"] = label_df["id"].astype(str)
label_series = label_df.set_index("id")["label"].astype(np.int8)

print(label_df["split"].value_counts())

# === 2. 与 combined_df 特征表对齐 ====================================
combined_df = combined_df.copy()
combined_df.index = combined_df.index.astype(str)

combined_unique = combined_df.loc[~combined_df.index.duplicated(keep="first")]

dataset = combined_unique.join(label_series, how="inner")

dataset = dataset.loc[~dataset.index.duplicated(keep="first")]

print(dataset.shape)
print(dataset.index.nunique())
print(
    "缺少的 valid ids 数量：",
    np.setdiff1d(valid_df["id"].astype(str).unique(), dataset.index).size
)

# === 3. 按官方 ID 划分 train / valid ==================================
train_ids = np.intersect1d(train_df["id"].astype(str).unique(), dataset.index)
valid_ids = np.intersect1d(valid_df["id"].astype(str).unique(), dataset.index)

X_train = dataset.loc[train_ids].drop(columns="label")
y_train = dataset.loc[train_ids, "label"]

X_val = dataset.loc[valid_ids].drop(columns="label")
y_val = dataset.loc[valid_ids, "label"]

print(f"Train shape: {X_train.shape}, positives = {y_train.sum()}")
print(f"Valid shape: {X_val.shape}, positives = {y_val.sum()}")

# === 4. 构建 Kaggle 测试集特征矩阵 X_test ==============================
combined_unique = combined_df.loc[~combined_df.index.duplicated(keep="first")].copy()
combined_unique.index = combined_unique.index.astype(str)

print(combined_unique.shape)

test_df_unique = (
    test_df.copy()
           .assign(id=test_df["id"].astype(str))
           .loc[~test_df["id"].duplicated(keep="first")]
)
test_ids = test_df_unique["id"].tolist()
print(f"Unique test ids: {len(test_ids)}")

assert combined_unique.index.is_unique, "combined_unique 仍然有重复索引，请先检查 combined_df"

X_test = (
    combined_unique.reindex(test_ids)
                   .reindex(columns=X_train.columns)
)

print(f"X_test shape: {X_test.shape}")
print(f"Missing test ids count: {(X_test.index.isna()).sum()}")

split
train    49451
valid    16721
Name: count, dtype: int64
(11022, 466)
11022
缺少的 valid ids 数量： 0
Train shape: (8234, 465), positives = 1449
Valid shape: (2788, 465), positives = 481
(14532, 465)
Unique test ids: 2741
X_test shape: (2741, 465)
Missing test ids count: 0


In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
XGBoost training script with extended hyperparameters.

Workflow:
1. Train/validate split → tune with early stopping → evaluate.
2. Refit on (train ∪ valid) using the best iteration count.
3. Predict on test, save outputs and artifacts.

Prerequisites: X_train, y_train, X_val, y_val, X_test already prepared.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
import xgboost as xgb
from xgboost.callback import EarlyStopping, LearningRateScheduler

# -------------------------------------------------------------------------
# 0. Basic preprocessing (shared by both stages)
# -------------------------------------------------------------------------
FILL_VALUE = 0.0  # replace with other imputation strategies if desired

X_train_filled = X_train.fillna(FILL_VALUE)
X_val_filled   = X_val.fillna(FILL_VALUE)
X_test_filled  = X_test.fillna(FILL_VALUE)

dtrain = xgb.DMatrix(X_train_filled, label=y_train)
dvalid = xgb.DMatrix(X_val_filled, label=y_val)
dtest  = xgb.DMatrix(X_test_filled)

# -------------------------------------------------------------------------
# 1. Parameter setup
# -------------------------------------------------------------------------
neg = float((y_train == 0).sum())
pos = float((y_train == 1).sum())
scale_pos_weight = neg / pos if pos > 0 else 1.0

params = {
    "objective": "binary:logistic",
    "eval_metric": ["auc", "logloss"],
    "tree_method": "hist",            # switch to "gpu_hist" if GPU is available
    "grow_policy": "lossguide",
    "max_depth": 7,
    "max_leaves": 128,
    "min_child_weight": 3.0,
    "gamma": 0.2,
    "max_delta_step": 1.0,
    "learning_rate": 0.04,
    "subsample": 0.75,
    "sampling_method": "uniform",     # gradient_based requires GPU
    "colsample_bytree": 0.65,
    "colsample_bylevel": 0.8,
    "colsample_bynode": 0.8,
    "lambda": 2.0,
    "alpha": 0.1,
    "scale_pos_weight": scale_pos_weight,
    "max_bin": 384,
    "monotone_constraints": "()",
    "verbosity": 1,
    "tree_method": "hist", # 如果有 GPU，改为 "gpu_hist"
    "device": "cuda", 
}

def lr_schedule(epoch: int) -> float:
    base_lr = params["learning_rate"]
    decay = 0.995
    min_lr = 0.01
    return max(base_lr * (decay ** epoch), min_lr)

callbacks = [
    EarlyStopping(
        rounds=75,
        metric_name="auc",
        data_name="valid",
        save_best=True,
    ),
    LearningRateScheduler(lr_schedule),
]

# -------------------------------------------------------------------------
# 2. Initial training with validation for early stopping
# -------------------------------------------------------------------------
evals = [(dtrain, "train"), (dvalid, "valid")]

bst = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=4000,
    evals=evals,
    callbacks=callbacks,
    verbose_eval=50,
)

best_iteration = bst.best_iteration or bst.num_boosted_rounds() - 1

train_pred = bst.predict(dtrain, iteration_range=(0, best_iteration + 1))
valid_pred = bst.predict(dvalid, iteration_range=(0, best_iteration + 1))

train_auc = roc_auc_score(y_train, train_pred)
valid_auc = roc_auc_score(y_val, valid_pred)

print(f"[Validation stage] Train AUC: {train_auc:.4f}")
print(f"[Validation stage] Valid AUC: {valid_auc:.4f}")
print(f"[Validation stage] Best iteration: {best_iteration}")
print(f"[Validation stage] Best valid AUC: {bst.attributes().get('best_score', 'N/A')}")

valid_prob_df = (
    pd.DataFrame({
        "id": X_val.index.astype(str),
        "pred_prob": valid_pred
    })
    .sort_values("id")
    .reset_index(drop=True)
)
valid_prob_df.to_csv("valid_pred_probs_xgb_full.csv", index=False)

# -------------------------------------------------------------------------
# 3. Refit on train ∪ valid using best iteration count
# -------------------------------------------------------------------------
X_full_filled = pd.concat([X_train_filled, X_val_filled], axis=0)
y_full = pd.concat([y_train, y_val], axis=0)

neg_full = float((y_full == 0).sum())
pos_full = float((y_full == 1).sum())
scale_pos_weight_full = neg_full / pos_full if pos_full > 0 else 1.0

params_full = params.copy()
params_full["scale_pos_weight"] = scale_pos_weight_full

dtrain_full = xgb.DMatrix(X_full_filled, label=y_full)

final_rounds = (best_iteration + 1) if best_iteration is not None else 4000

bst_full = xgb.train(
    params=params_full,
    dtrain=dtrain_full,
    num_boost_round=final_rounds,
    verbose_eval=50,
)

# -------------------------------------------------------------------------
# 4. Test inference with the refitted model
# -------------------------------------------------------------------------
test_pred_full = bst_full.predict(dtest)

submission = pd.DataFrame({
    "id": X_test.index.astype(str),
    "readmitted_within_30days": test_pred_full
})
submission.to_csv("submission_xgb_fulltrain.csv", index=False)

bst_full.save_model("xgb_readmit_fulltrain.json")

importance_gain = bst_full.get_score(importance_type="gain")
importance_cover = bst_full.get_score(importance_type="cover")
importance_weight = bst_full.get_score(importance_type="weight")

importance_df = (
    pd.DataFrame([
        {
            "feature": feat,
            "gain": importance_gain.get(feat, 0.0),
            "cover": importance_cover.get(feat, 0.0),
            "weight": importance_weight.get(feat, 0.0),
        }
        for feat in set(importance_gain) | set(importance_cover) | set(importance_weight)
    ])
    .sort_values("gain", ascending=False)
)

importance_df.to_csv("xgb_feature_importance_fulltrain.csv", index=False)

print("Refit on combined train+valid completed. Test predictions saved to submission_xgb_fulltrain.csv.")

NameError: name 'X_train' is not defined